# 🧠 Integración de Voz en Sistemas Expertos
## Lógica Simbólica y Diálogo

Un **Sistema Experto** emula la toma de decisiones de un humano. Al agregarle voz, lo hacemos más accesible.

**Componentes:**
1.  **Base de Conocimiento:** Reglas (Si tienes fiebre y tos -> Posible gripe).
2.  **Motor de Inferencia:** Lógica que cruza los datos del usuario con las reglas.
3.  **Interfaz:** En este caso, de voz.

In [8]:
import speech_recognition as sr
import pyttsx3
import time

# Configuración inicial simplificada
recognizer = sr.Recognizer()
engine = pyttsx3.init()
engine.setProperty('rate', 145)

## 1. Definición de la Clase del Sistema Médico

Vamos a encapsular nuestro sistema experto en una clase. Observa cómo la base de conocimiento (`self.sintomas_diagnosticos`) es un diccionario que mapea **tuplas de síntomas** a **diagnósticos**.

In [9]:
class SistemaExpertoMedico:
    def __init__(self):
        # Base de Conocimiento (Hechos y Reglas)
        self.sintomas_diagnosticos = {
            ('fiebre', 'tos', 'garganta'): 'Parece una gripe común.',
            ('fiebre', 'cabeza', 'rigidez'): 'Alerta: Posible meningitis. Ve al médico urgente.',
            ('estómago', 'náuseas'): 'Posible gastritis o infección estomacal.',
            ('cabeza', 'cansancio'): 'Podría ser deshidratación o estrés.'
        }
        self.sintomas_detectados = []
    
    def hablar(self, texto):
        print(f"🩺 Sistema: {texto}")
        engine.say(texto)
        engine.runAndWait()
    
    def escuchar(self):
        with sr.Microphone() as source:
            print("👂 Escuchando...")
            try:
                recognizer.adjust_for_ambient_noise(source, duration=0.5)
                audio = recognizer.listen(source, timeout=3)
                texto = recognizer.recognize_google(audio, language='es-ES')
                print(f"👤 Usuario: {texto}")
                return texto.lower()
            except:
                return ""

    def motor_de_inferencia(self):
        # Convertimos la lista de síntomas detectados a un conjunto para comparar
        # Nota: Esta es una inferencia simple. Sistemas reales usan lógica difusa o árboles.
        
        diagnostico_final = "No tengo suficiente información para un diagnóstico."
        
        # Buscamos coincidencias en la base de conocimiento
        for sintomas_clave, diagnostico in self.sintomas_diagnosticos.items():
            # Verificamos si TODOS los síntomas de la regla están presentes
            # all() devuelve True si todos los elementos son verdaderos
            coincidencias = 0
            for sintoma_requerido in sintomas_clave:
                if sintoma_requerido in self.sintomas_detectados:
                    coincidencias += 1
            
            # Umbral de decisión: Si tiene la mayoría de síntomas
            if coincidencias >= 2:
                diagnostico_final = diagnostico
                break
        
        return diagnostico_final

    def iniciar_consulta(self):
        self.hablar("Hola. Soy tu asistente médico. Describe tus síntomas. Di 'terminar' al finalizar.")
        
        activo = True
        while activo:
            respuesta = self.escuchar()
            
            if "terminar" in respuesta or "gracias" in respuesta:
                activo = False
                continue
            
            # Extracción simple de palabras clave (Keyword Spotting)
            palabras_clave = ['fiebre', 'tos', 'garganta', 'cabeza', 'rigidez', 'estómago', 'náuseas', 'cansancio']
            
            encontrado = False
            for palabra in palabras_clave:
                if palabra in respuesta:
                    if palabra not in self.sintomas_detectados:
                        self.sintomas_detectados.append(palabra)
                        self.hablar(f"Anotado: {palabra}.")
                        encontrado = True
            
            if not encontrado and respuesta != "":
                self.hablar("¿Tienes algún otro síntoma?")

        # Fase final: Diagnóstico
        resultado = self.motor_de_inferencia()
        self.hablar(f"Basado en tus síntomas ({', '.join(self.sintomas_detectados)}), mi conclusión es:")
        self.hablar(resultado)

## 2. Ejecución del Sistema
Ejecuta la celda de abajo y habla al micrófono. Di frases como: "Tengo mucha fiebre y me duele la cabeza".

In [ ]:
doctor_bot = SistemaExpertoMedico()
doctor_bot.iniciar_consulta()

🩺 Sistema: Hola. Soy tu asistente médico. Describe tus síntomas. Di 'terminar' al finalizar.
👂 Escuchando...
👂 Escuchando...
👂 Escuchando...
👤 Usuario: pues ya saben que él es este
🩺 Sistema: ¿Tienes algún otro síntoma?
👂 Escuchando...
👂 Escuchando...


## 🧠 Reto de Lógica
El sistema actual es muy estricto. Modifica el diccionario `sintomas_diagnosticos` para agregar una nueva enfermedad, por ejemplo, "Alergia" (estornudos, ojos rojos).